---
title: "2.6 Combining datasets safely"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/06-combining.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/06-combining.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [2.5 Group summaries: the pivot table lesson](../02-data/05-group-summaries.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Choose join keys and distinguish left from inner joins.
- Validate the expected one-to-one or many-to-one relationship.
- Audit unmatched rows and guard against multiplying observations.

## Why it matters

A merge attaches information from one table to matching observations in another. A **key** is the field, or set of fields, used to identify a match. A country-year panel can have many rows for a country while a country-region lookup should have only one. The most dangerous merge can run without an error and still multiply your observations. We use explicit validation and counts to detect that risk.

## Worked example

### Start with known matches

These made-up numbers describe fictional Northland and Eastria. The second table supplies one region label per country. A **left join** keeps every row from the table on the left and attaches matching fields from the right. `on` identifies the shared key.

In [ ]:
import pandas as pd
small = pd.DataFrame({"country": ["Northland", "Northland", "Eastria"], "year": [2002, 2007, 2007], "life_exp": [70.0, 74.0, 78.0]})
lookup = pd.DataFrame({"country": ["Northland", "Eastria"], "region": ["Northern group", "Eastern group"]})
small_joined = small.merge(lookup, on="country", how="left", validate="many_to_one", indicator=True)
small_joined

The result still has three observations. `many_to_one` means many left rows may share a country, but the right table must have only one row per country. `indicator=True` adds `_merge`, an audit column showing which rows matched.

### Make a non-match visible

An **inner join** keeps only rows with a match on both sides. Compare it with a left join using a deliberately incomplete lookup that contains only Northland.

In [ ]:
incomplete_lookup = lookup.loc[lookup["country"] == "Northland"].copy()
left_practice = small.merge(incomplete_lookup, on="country", how="left", validate="many_to_one", indicator=True)
inner_practice = small.merge(incomplete_lookup, on="country", how="inner", validate="many_to_one")
print(left_practice["_merge"].value_counts())
print("Left rows:", len(left_practice))
print("Inner rows:", len(inner_practice))

The left join keeps all three rows and marks Eastria as left_only; the inner join has only two rows. Losing an unmatched country can silently change your sample. Choose the join to fit the question and inspect non-matches before accepting it.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Prepare the real lookup

The course separates the extract's existing region labels into `country_regions.csv`. It is not a new dataset or an independent classification; it is the same country/region information stored as a lookup for practice. In a fresh Colab session, the cell rebuilds that identical lookup from the loaded panel. `.drop_duplicates()` keeps each distinct country/region pair once.

In [ ]:
countries = pd.read_csv(data_file)
regions_file = Path("data/country_regions.csv")
if not regions_file.exists():
    countries[["country", "region"]].drop_duplicates().sort_values("country").to_csv(regions_file, index=False, lineterminator="\n")
country_regions = pd.read_csv(regions_file)
print(country_regions.shape)
print("Repeated country keys:", country_regions["country"].duplicated().sum())
country_regions.head()

The lookup has 142 rows, two columns and no repeated country key. A repeated key with conflicting labels needs investigation, not an automatic drop of whichever row comes second. The [data documentation](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md) describes the deterministic derivation.

### Join and verify the panel

Remove the existing region field in a practice table, then join it back from the lookup. `.drop(columns=...)` removes named fields from the returned table. Use a left join so no country-years are silently dropped; validate the key relationship and count the match statuses.

In [ ]:
panel_without_region = countries.drop(columns="region")
joined = panel_without_region.merge(country_regions, on="country", how="left", validate="many_to_one", indicator=True)
print("Before:", len(panel_without_region))
print("After:", len(joined))
print(joined["_merge"].value_counts())

All 1,704 country-years remain and all are matched. This is expected because both tables came from the same extract. With an independent source, spelling changes, missing keys or historical borders can create non-matches even when the merge command is correct.

### Inspect the audit before analysis

Select any left-only observations and display their distinct country names. A zero-row result is a useful finding here. Keep the audit information until you have checked it.

In [ ]:
unmatched = joined.loc[joined["_merge"] == "left_only", ["country", "year"]]
print("Unmatched country-years:", len(unmatched))
print(unmatched["country"].unique().tolist())

The count is zero and the list is empty. Many-to-one validation checks uniqueness on the right; it does not guarantee that every key matched or that the labels are substantively correct. Those are separate checks. In pandas, null keys can match other null keys, so inspect missing key values before a real merge.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Match one year one to one

Filter the country panel to 2007, remove region, and merge the lookup using `validate="one_to_one"`. Print the result size.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
cross_section = countries.loc[countries["year"] == 2007].drop(columns="region")
joined_2007 = cross_section.merge(country_regions, on="country", how="left", validate="one_to_one", indicator=True)
print(joined_2007.shape)

There are 142 observations and seven columns including the audit field. One-to-one is appropriate after selecting one year because each country now appears once on both sides.

</details>

### 2. Diagnose an incomplete lookup

Use `left_practice` from the fictional example to list unmatched countries. Do not drop them silently.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
unmatched_fictional = left_practice.loc[left_practice["_merge"] == "left_only", "country"].tolist()
print(unmatched_fictional)

The list contains Eastria. Decide whether its absent region is a spelling error, missing lookup information, or an intended limitation before changing the sample.

</details>

### 3. Use a compound key

Create a second real-data practice table with country, year and population. Merge it with country, year and life expectancy using both country and year as the key. Validate one-to-one and print its row count.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
life_table = countries[["country", "year", "life_exp"]]
population_table = countries[["country", "year", "population"]]
compound_join = life_table.merge(population_table, on=["country", "year"], how="left", validate="one_to_one")
print(len(compound_join))

The 1,704 country-year pairs match one-to-one. Country alone would not identify an observation on either side of these two panel tables.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### The lookup has duplicate keys

Deliberately append the first fictional lookup row a second time, then request a many-to-one merge. `pd.concat` places tables beneath one another; it does not match their keys.

In [ ]:
try:
    duplicate_lookup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)
    small.merge(duplicate_lookup, on="country", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(f"{type(error).__name__}: {error}")

The merge rejects non-unique right keys. Without validation, both Northland observations would be copied for each matching lookup row. Investigate whether duplication is accidental or the key needs more fields.

### A one-to-one claim is wrong for a panel

Here the right keys are unique, but the left country key repeats across years.

In [ ]:
try:
    panel_without_region.merge(country_regions, on="country", validate="one_to_one")
except pd.errors.MergeError as error:
    print(f"{type(error).__name__}: {error}")

Use many_to_one for this lookup, or include year on both sides when joining two panel datasets. Do not weaken validation to many_to_many merely to make an error disappear.

## Recap

- Identify the row meaning and key on each side before merging.
- A left join preserves left rows; an inner join excludes non-matches.
- Validate the intended key relationship and check row counts.
- Inspect unmatched and missing keys separately from duplicate keys.

**Next:** [2.7 Wide and long data; panel structure](../02-data/07-reshaping.ipynb).

::: {.callout-note title="Coming from Stata / R / Excel?"}
Stata merge relationships such as m:1 correspond to pandas validation such as many_to_one. pandas does not automatically enforce your intended relationship unless you request validation.
:::